# Model Tester

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

## Setup

In [ ]:
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
# os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.losses import MeanSquaredError, mse
from tensorflow.keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, LearningRateSchedule

from scripts.utils import Config, setup_logging, load_data
from scripts.utils.plots import plot_histogram, plot_predictions
from scripts.utils.tf.dataloaders import *
from scripts.utils.tf.plots import plot_metrics
from scripts.utils.tf.callbacks import TimedLoggingCallback, WarmupLearningRate

from scripts.trainer import *

In [ ]:
logger = setup_logging(__name__, level=logging.INFO)
logging.getLogger("scripts").setLevel(logging.DEBUG)
logging.getLogger("tensorflow").setLevel(logging.ERROR)

In [ ]:
class CustomSchedule(LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.d_model = d_model
        self.d_model = tf.cast(self.d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps**-1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

## Configure

In [ ]:
# s = Config(["settings/planck.json", "--no-lensing", "--no-noise", "--narray", "500"])
s = Config(
    [
        "settings/planck.json",
        "--lensing",
        "--noise",
        "--narray",
        "50",
        "--base_name",
        "planck_lowfnl",
        "--fnl_range",
        "-50",
        "50",
    ]
)

MAX_EPOCHS = 100
BATCH_SIZE = 8

# just some info for the model name
timestamp = int(time.time())
model_settings = {
    # "dropout_rate": 0.1,
    "name": f"tester-{s.base_name}-{timestamp}",
}

data_loader_args = {
    "shuffle": True,
    "shuffle_buffer": 1000,
    "seed": None,
    "batch_size": BATCH_SIZE,
    "cache": True,
    "normalize": True,
}

# additional metrics we are interested in
metrics = ["mean_absolute_error"]

callbacks = [
    # EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    # TimedLoggingCallback(print_frequency=3),
    # TensorBoard(log_dir=f"{s.tb_dir}/{model_settings['name']}"),
    TerminateOnNaN(),
]

In [ ]:
# import wandb
# from wandb.keras import WandbMetricsLogger

# # wandb.tensorboard.patch(root_logdir=s.tb_dir)

# wandb.init(
#     project="mlpng",
#     tags=["attn-alm", "dev"],
#     config=s.settings | model_settings,
#     dir="data",
#     sync_tensorboard=True,
# )

# callbacks.append(WandbMetricsLogger())

## Model 1

In [ ]:
data_loader = AlmLoader(
    s.alm_file,
    channels_last=True,
    **data_loader_args,
).as_tfds(auto_convert=True)

train_dataset, test_dataset, val_dataset = data_loader.get_split(0.8, 0.1, 0.1)
learning_rate = CustomSchedule(1024)
# learning_rate = WarmupLearningRate()
# learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model = alm_modelV1_2(Input(data_loader.shape), **model_settings)  # type: ignore
    model.compile(optimizer=opt, loss=mse, metrics=metrics)

model.summary()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

try:
    with h5py.File(s.alm_file, "r", swmr=True, locking=False) as hdf:
        fisher = hdf.get("fisher", [None])[0]
        logger.info(f"Loaded fisher matrix: {fisher}")
except Exception as e:
    logger.error(f"Could not load fisher matrix: {e}")
    fisher = None

y_pred = model.predict(test_dataset, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_dataset])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## Patch Model

In [ ]:
data_loader = PatchLoader(
    s.patch_file,
    # channels_last=True,
    **data_loader_args,
).as_tfds(auto_convert=True)

train_dataset, test_dataset, val_dataset = data_loader.get_split(0.8, 0.1, 0.1)
learning_rate = CustomSchedule(1024)
# learning_rate = WarmupLearningRate()
# learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model = patch_modelV1(Input(data_loader.shape), **model_settings)  # type: ignore
    model.compile(optimizer=opt, loss=mse, metrics=metrics)

model.summary()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

try:
    with h5py.File(s.alm_file, "r", swmr=True, locking=False) as hdf:
        fisher = hdf.get("fisher", [None])[0]
        logger.info(f"Loaded fisher matrix: {fisher}")
except Exception as e:
    logger.error(f"Could not load fisher matrix: {e}")
    fisher = None

y_pred = model.predict(test_dataset, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_dataset])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)